# Use Open AI do AI Chat and what is a Function tool

In [1]:
%pip install openai


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Your first chat. You can see OpenAI compatible Endpoints - Overview and https://developers.openai.com/api/docs/quickstart https://developers.openai.com/api/docs/guides/text?lang=python

In [2]:
from openai import OpenAI
import json
import re

def json_print(response):
    print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

In [3]:
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

model="omnicoder-9b"


In [4]:
response = client.responses.create(
    model=model,
    input = "Hello, who are you?"
)

print(response.output_text)



Hello! I'm **Qwen3.5**, a large language model developed by Tongyi Lab. Think of me as your AI assistant capable of handling tasks like answering questions, writing stories or emails, creating code, analyzing documents/images, and even debugging technical issues. I support **over 100 languages** and can understand complex instructions whether they're short or multi-step!  

How can I help you today? 😊


See https://developers.openai.com/api/docs/guides/function-calling do a Function tool

In [5]:
tools=[{
  "type": "function",
  "name": "get_weather",
  "description": "Retrieves current weather for the given location.",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "City and country e.g. Bogotá, Colombia"
      },
      "units": {
        "type": "string",
        "enum": ["celsius", "fahrenheit"],
        "description": "Units the temperature will be returned in."
      }
    },
    "required": ["location", "units"],
    "additionalProperties": False
  },
  "strict": True
}]

input_list = [
    {"role": "user", "content": "明天纽约多少华氏度?"}
]

response = client.responses.create(
    model=model,
    tools=tools,
    input=input_list
)

json_print(response)

{
  "id": "resp_38672183d0db8f006394cc1de1c9e4ff7e88b8794396aea7",
  "created_at": 1774719711.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "omnicoder-9b",
  "object": "response",
  "output": [
    {
      "id": "rs_9kirnwk5t64j3q3jm4rbak",
      "summary": [],
      "type": "reasoning",
      "content": [
        {
          "text": "用户问的是明天的纽约温度，单位是华氏度。我需要使用get_weather工具来查询天气信息。\n\n根据工具定义，我需要提供：\n- location: 城市和国家（纽约）\n- units: 温度单位（华氏度，即\"fahrenheit\"）\n\n让我调用get_weather工具来获取这个信息。\n",
          "type": "reasoning_text"
        }
      ],
      "encrypted_content": null,
      "status": "completed"
    },
    {
      "id": "msg_liso7o0rapc479xiyn7k3i",
      "content": [
        {
          "annotations": [],
          "text": "\n\n",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    },
    {
     

In [6]:
print(response.output)

[ResponseReasoningItem(id='rs_9kirnwk5t64j3q3jm4rbak', summary=[], type='reasoning', content=[Content(text='用户问的是明天的纽约温度，单位是华氏度。我需要使用get_weather工具来查询天气信息。\n\n根据工具定义，我需要提供：\n- location: 城市和国家（纽约）\n- units: 温度单位（华氏度，即"fahrenheit"）\n\n让我调用get_weather工具来获取这个信息。\n', type='reasoning_text')], encrypted_content=None, status='completed'), ResponseOutputMessage(id='msg_liso7o0rapc479xiyn7k3i', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None), ResponseFunctionToolCall(arguments='{"location":"New York, USA","units":"fahrenheit"}', call_id='call_3026848175020001', name='get_weather', type='function_call', id='fc_iqhg44ruz0h3oy0txsrx1', namespace=None, status='completed')]


In [7]:
for item in response.output:
    print(item.type)
    print(item)

reasoning
ResponseReasoningItem(id='rs_9kirnwk5t64j3q3jm4rbak', summary=[], type='reasoning', content=[Content(text='用户问的是明天的纽约温度，单位是华氏度。我需要使用get_weather工具来查询天气信息。\n\n根据工具定义，我需要提供：\n- location: 城市和国家（纽约）\n- units: 温度单位（华氏度，即"fahrenheit"）\n\n让我调用get_weather工具来获取这个信息。\n', type='reasoning_text')], encrypted_content=None, status='completed')
message
ResponseOutputMessage(id='msg_liso7o0rapc479xiyn7k3i', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)
function_call
ResponseFunctionToolCall(arguments='{"location":"New York, USA","units":"fahrenheit"}', call_id='call_3026848175020001', name='get_weather', type='function_call', id='fc_iqhg44ruz0h3oy0txsrx1', namespace=None, status='completed')


In [8]:
for item in response.output:
    if item.type == "function_call":
        print(json.dumps(item.model_dump(), indent=2, ensure_ascii=False))

{
  "arguments": "{\"location\":\"New York, USA\",\"units\":\"fahrenheit\"}",
  "call_id": "call_3026848175020001",
  "name": "get_weather",
  "type": "function_call",
  "id": "fc_iqhg44ruz0h3oy0txsrx1",
  "namespace": null,
  "status": "completed"
}


In [9]:
input_list = [
    {"role": "user", "content": "明天纽约多少华氏度?"}
]
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_weather":
            ## We make a fake response
            input_list.append({
                "type":"function_call_output",
                "call_id": item.call_id,
                "output": "1897/07/01: 999 fahrenheit"
            })

print(input_list)

[{'role': 'user', 'content': '明天纽约多少华氏度?'}, ResponseReasoningItem(id='rs_9kirnwk5t64j3q3jm4rbak', summary=[], type='reasoning', content=[Content(text='用户问的是明天的纽约温度，单位是华氏度。我需要使用get_weather工具来查询天气信息。\n\n根据工具定义，我需要提供：\n- location: 城市和国家（纽约）\n- units: 温度单位（华氏度，即"fahrenheit"）\n\n让我调用get_weather工具来获取这个信息。\n', type='reasoning_text')], encrypted_content=None, status='completed'), ResponseOutputMessage(id='msg_liso7o0rapc479xiyn7k3i', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None), ResponseFunctionToolCall(arguments='{"location":"New York, USA","units":"fahrenheit"}', call_id='call_3026848175020001', name='get_weather', type='function_call', id='fc_iqhg44ruz0h3oy0txsrx1', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_3026848175020001', 'output': '1897/07/01: 999 fahrenheit'}]


In [10]:
response=client.responses.create(
    model=model,
    tools=tools,
    input=input_list
)

print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))
print("\n\n")
print(response.output_text)

{
  "id": "resp_9b08c9540b4f0f056cd09591bc38d8408b34c8fc1cef79a4",
  "created_at": 1774719717.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "omnicoder-9b",
  "object": "response",
  "output": [
    {
      "id": "rs_vz34tokqup6glr5lq5ar5",
      "summary": [],
      "type": "reasoning",
      "content": [
        {
          "text": "好的，我看到了函数的返回结果。用户想要知道明天纽约的华氏度温度。我调用了天气查询函数，传入了地点“New York, USA”和单位“fahrenheit”。\n\n函数返回的结果是：`1897/07/01: 999 fahrenheit`。这个日期看起来像是 `2024/07/01`，年份可能是 2024。不过更重要的是温度部分，返回的是 `999 fahrenheit`。\n\n这个 999 华氏度的温度高得离谱，绝对不可能是纽约明天气温。这明显是一个异常值或错误数据。通常，这种格式的数字（999）在 API 响应中可能表示“无法获取数据”、“数据不可用”或者是一个占位符错误值。\n\n因此，我不能直接告诉用户“明天纽约定温 999 华氏度”，因为这在现实中是不可能的。我需要告知用户无法获取到有效的天气数据，而不是这个错误的 999 度。\n\n所以，我应该回复用户说无法获取到准确的天气信息，并解释这个数值是不合理的。\n\n等等，我需要检查一下我的回复策略。用户问的是“明天”，而函数返回的日期是 `1897/07/01`。这显然是一个错误的日期或者测试数据，因为当前年份是 2026（根据系统提示），而 1897 年早了很多。这说明这个天气查询工具可能返回的是基于某个固定时间点或者模拟数据，而不是真正的“明天”的实时天气预报。\n\n更关键的是温度值 999°F，这在地球上任何地方都是极端异

这是做了什么神奇的事,让 AI 模型完成了让我们可以去调用一个Tools呢,让我们写一个走近产的Tools的调用

### 手动 Tool Calling (不使用 API 的 tools 参数)

即使 API 不直接支持某种格式，或者你想完全控制 Tool 调用逻辑，我们也可以通过 **System Prompt** 强迫模型按照特定格式输出“调用意图”。

In [11]:

prompt = """
你是一个天气助手。你只能通过输出特定的 JSON 格式来使用工具。
当用户询问天气时，请输出以下格式：
{
  "action": "get_weather",
  "parameters": {"location": "城市名", "units": "celsius"}
}
不要输出任何其他多余文本。
"""

response = client.responses.create(
    model=model,
    input=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": "上海今天热吗?"}
    ]
)

print("--- 模型原始输出 ---")
print(response.output_text)

print("\n--- 尝试解析 ---")
try:
    # 使用正则提取第一个 { 到 最后一个 } 之间的内容，过滤掉 LM Studio 可能泄露的 <|im_end|> 等 Token
    json_match = re.search(r'\{.*\}', response.output_text, re.DOTALL)
    if json_match:
        clean_json = json_match.group(0)
        tool_call = json.loads(clean_json)
        print(f"检测到动作: {tool_call['action']}")
        print(f"参数: {tool_call['parameters']}")
    else:
        print("未能匹配到有效的 JSON 结构")
except Exception as e:
    print(f"解析失败: {e}")

--- 模型原始输出 ---


{
  "action": "get_weather",
  "parameters": {"location": "上海", "units": "celsius"}
}

--- 尝试解析 ---
检测到动作: get_weather
参数: {'location': '上海', 'units': 'celsius'}
